# Spatial relationships and operations · 2026
Based on the original scipy2018-geospatial teaching sequence.

Sources: ISTAT administrative boundaries, **1 January 2026**, and Ministry of Health **national pharmacy CSV, 22 October 2026**.

Inspect a table, create points, align CRS, evaluate predicates, perform spatial joins, buffer, simplify, clip, dissolve and overlay.

Keep source records and disclose assumptions. The pharmacy catalogue includes branches and dispensaries. A valid record and a distinct pharmacy are different counting units.

# SETUP

### Block 01 · Learning goal and AI prompt

**Learning objective:** Separate environment setup from analysis.

**Check after execution:** Did installation finish successfully?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Install geopandas, matplotlib, requests, folium and mapclassify in this notebook using a notebook-aware pip command. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
%pip install -q geopandas matplotlib requests folium mapclassify

## Import

### Block 02 · Learning goal and AI prompt

**Learning objective:** Prepare the libraries needed for spatial integration.

**Check after execution:** Which operations will each library support?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Import pandas as pd, geopandas as gpd, requests, matplotlib.pyplot as plt and Path from pathlib. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
from pathlib import Path

## ISTAT 2026 administrative units
Read the original national ZIP and build a multilayer GeoPackage locally. There is no invented “2026 course GeoPackage” URL.

https://www.istat.it/notizia/confini-delle-unita-amministrative-a-fini-statistici-al-1-gennaio-2018-2/

### Block 03 · Learning goal and AI prompt

**Learning objective:** Build a reusable multilayer dataset from original sources.

**Check after execution:** Which layers and CRS are stored in the file?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Download and extract the official ISTAT 2026 generalised ZIP, define the same source variables and read_istat helper as notebook 01, and create a local GeoPackage named istat_administrative_units_generalized_2026.gpkg in DATA. Map RipGeo to macroregions, Reg to regions, ProvCM to provinces and Com to municipalities. Reuse an existing complete file. Store its path as geopackage_istat_file and url. Use the official 2026 ZIP URL, not an invented remote GeoPackage link. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import requests
from pathlib import Path
from zipfile import ZipFile

YEAR = 2026
ISTAT_URL = ("https://www.istat.it/storage/cartografia/"
             "confini_amministrativi/generalizzati/2026/Limiti01012026_g.zip")
DATA = Path("data_2026").resolve()
DATA.mkdir(exist_ok=True)
archive = DATA / "Limiti01012026_g.zip"
if not archive.exists():
    response = requests.get(ISTAT_URL, timeout=120)
    response.raise_for_status()
    archive.write_bytes(response.content)
source_dir = DATA / "istat"
if not list(source_dir.rglob("*.shp")):
    with ZipFile(archive) as z:
        z.extractall(source_dir)

def read_istat(prefix):
    candidates = list(source_dir.rglob(prefix + "*.shp"))
    if len(candidates) != 1:
        raise ValueError(f"Expected one {prefix} layer, found {candidates}")
    return gpd.read_file(candidates[0])

# Build the multilayer GeoPackage from the original 2026 Shapefiles.
geopackage_istat_file = str(DATA / "istat_administrative_units_generalized_2026.gpkg")
if not Path(geopackage_istat_file).exists():
    for layer, prefix in {"macroregions": "RipGeo", "regions": "Reg",
                          "provinces": "ProvCM", "municipalities": "Com"}.items():
        read_istat(prefix).to_file(geopackage_istat_file, layer=layer, driver="GPKG")
url = geopackage_istat_file


### Block 04 · Learning goal and AI prompt

**Learning objective:** Load the intended statistical division layer.

**Check after execution:** Is this layer a statistical division or a municipality layer?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Read the macroregions layer of geopackage_istat_file into macroregions. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
macroregions = gpd.read_file(geopackage_istat_file, layer="macroregions")

you can repeat the operation ( = direct download) for each layer but it's better to download the file once and load the layers step by step.

### Block 05 · Learning goal and AI prompt

**Learning objective:** Verify the contents of a cached multilayer file.

**Check after execution:** Are all four expected administrative levels available?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Print the layer list from geopackage_istat_file using GeoPandas. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
print(gpd.list_layers(geopackage_istat_file))

### Block 06 · Learning goal and AI prompt

**Learning objective:** Load region geometry and attributes.

**Check after execution:** How does this observation unit differ from macroregions?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Read the regions layer of geopackage_istat_file into regions. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
regions = gpd.read_file(geopackage_istat_file,layer="regions")

### Block 07 · Learning goal and AI prompt

**Learning objective:** Load province geometry and attributes.

**Check after execution:** Which fields identify the province?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Read the provinces layer of geopackage_istat_file into provinces. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
provinces = gpd.read_file(geopackage_istat_file,layer="provinces")

### Block 08 · Learning goal and AI prompt

**Learning objective:** Load municipalities for later point–polygon matching.

**Check after execution:** Which fields contain municipality names and stable identifiers?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Read the municipalities layer of geopackage_istat_file into municipalities. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
municipalities = gpd.read_file(geopackage_istat_file,layer="municipalities")

### Block 09 · Learning goal and AI prompt

**Learning objective:** Inspect the macroregion observations.

**Check after execution:** Does one row correspond to one statistical macroregion?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display macroregions. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
macroregions

## Pharmacies: start with the national source
Catalogue: https://www.dati.salute.gov.it/it/dataset/farmacie/

The CSV is a dated national snapshot, not a prepared Trentino subset. Read the linked data dictionary. Licence: IODL 2.0. Some published coordinates: © OpenStreetMap contributors.

Historical rows must be filtered for a reference date before counting.

### Block 10 · Learning goal and AI prompt

**Learning objective:** Inspect a national source before creating a local subset.

**Check after execution:** Why should identifiers and historical dates be inspected before conversion?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Download the Ministry national pharmacy CSV snapshot 2026-10-22 using requests and HTTP error checking. Cache it in DATA as FRM_FARMA_5_20261022.csv. Read it with semicolon delimiter, UTF-8-sig and all fields as strings into pharmacies_raw. Set PHARMACIES_CATALOGUE, PHARMACIES_URL, REFERENCE_DATE and pharmacy_csv. Use https://www.dati.salute.gov.it/sites/default/files/opendata/FRM_FARMA_5_20261022.csv . Display the first rows. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
PHARMACIES_CATALOGUE = "https://www.dati.salute.gov.it/it/dataset/farmacie/"
PHARMACIES_URL = "https://www.dati.salute.gov.it/sites/default/files/opendata/FRM_FARMA_5_20261022.csv"
REFERENCE_DATE = pd.Timestamp("2026-10-22")
pharmacy_csv = DATA / "FRM_FARMA_5_20261022.csv"
if not pharmacy_csv.exists():
    response = requests.get(PHARMACIES_URL, timeout=120)
    response.raise_for_status()
    pharmacy_csv.write_bytes(response.content)
# Keep identifiers as strings and retain the national source unchanged.
pharmacies_raw = pd.read_csv(pharmacy_csv, sep=";", dtype=str, encoding="utf-8-sig")
pharmacies_raw.head()


### investigate the data

### Block 11 · Learning goal and AI prompt

**Learning objective:** Distinguish a source record from a currently valid pharmacy.

**Check after execution:** Could historical rows make this larger than the number of active pharmacies?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Print the number of rows in pharmacies_raw with a descriptive label. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
print("National source rows:", len(pharmacies_raw))

### Block 12 · Learning goal and AI prompt

**Learning objective:** Inspect actual source values before cleaning.

**Check after execution:** How are missing values, dates and coordinates represented?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display the first rows of pharmacies_raw. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies_raw.head()

### Block 13 · Learning goal and AI prompt

**Learning objective:** Make the observation unit and time selection explicit.

**Check after execution:** What does a record mean after the time filter, and which date convention was assumed?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Rename the source fields using columns: cod_farmacia to pharmacy_id, cod_comune to municipality_code, comune to municipality, provincia to province, cod_provincia to province_code, indirizzo to address, latitudine to latitude, longitudine to longitude. Work on a copy named pharmacies and preserve the source row index as record_id. Parse validity dates using day/month/year, flag invalid_dates and print their count. Keep rows valid on REFERENCE_DATE with inclusive start and end dates; a dash means no end date. Do not silently accept unparseable dates or deduplicate IDs. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
columns = {"cod_farmacia": "pharmacy_id", "cod_comune": "municipality_code",
           "comune": "municipality", "provincia": "province", "cod_provincia": "province_code",
           "indirizzo": "address", "latitudine": "latitude", "longitudine": "longitude"}
pharmacies = pharmacies_raw.rename(columns=columns).copy()
pharmacies["record_id"] = pharmacies.index
start = pd.to_datetime(pharmacies.data_inizio_validita, format="%d/%m/%Y", errors="coerce")
end = pd.to_datetime(pharmacies.data_fine_validita.replace("-", pd.NA),
                     format="%d/%m/%Y", errors="coerce")
invalid_dates = start.isna() | (pharmacies.data_fine_validita.ne("-") & end.isna())
print("Rows requiring date review:", invalid_dates.sum())
# Explicit convention: start and end dates are inclusive; '-' means no end date.
active = (~invalid_dates & start.le(REFERENCE_DATE)
          & (end.isna() | end.ge(REFERENCE_DATE)))
pharmacies = pharmacies.loc[active].copy()


### Block 14 · Learning goal and AI prompt

**Learning objective:** Separate numeric parsing from evidence of positional correctness.

**Check after execution:** Do plausible numeric ranges prove a point is in the correct place?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Convert longitude and latitude in pharmacies from decimal-comma strings to numeric values, treating dashes and unparseable values as missing. Build coordinate_ok using longitude between -180 and 180 and latitude between -90 and 90. Preserve separate copies pharmacies_missing_coordinates and pharmacies_with_coordinates. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
# Decimal commas and '-' missing values occur in the original CSV.
for col in ["longitude", "latitude"]:
    pharmacies[col] = pd.to_numeric(
        pharmacies[col].str.replace(",", ".", regex=False).replace("-", pd.NA),
        errors="coerce")
coordinate_ok = pharmacies.longitude.between(-180, 180) & pharmacies.latitude.between(-90, 90)
pharmacies_missing_coordinates = pharmacies.loc[~coordinate_ok].copy()
pharmacies_with_coordinates = pharmacies.loc[coordinate_ok].copy()


### Block 15 · Learning goal and AI prompt

**Learning objective:** Audit selection and missingness before mapping.

**Check after execution:** Are records, IDs and mapped points the same denominator?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Create tn_records by selecting province_code 022 after padding to three digits. Print record count, distinct pharmacy IDs, the number of records without usable coordinates, and the national count of active rows whose pharmacy IDs occur more than once. Do not drop such rows automatically. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
tn_records = pharmacies[pharmacies.province_code.str.zfill(3).eq("022")].copy()
print("Valid Trentino records:", len(tn_records))
print("Distinct pharmacy IDs:", tn_records.pharmacy_id.nunique())
print("Trentino records lacking usable coordinates:",
      (~(tn_records.longitude.between(-180,180) & tn_records.latitude.between(-90,90))).sum())
print("Active IDs with several rows:", pharmacies.pharmacy_id.duplicated(keep=False).sum())

### Coordinate assumption
For this exercise, interpret the Ministry longitude/latitude fields as **WGS84 / EPSG:4326**. Verify coordinate metadata for a production use. Numeric range checks detect some errors but do not establish a CRS or prove positional accuracy. Keep records without coordinates in a separate audit table.

### Block 16 · Learning goal and AI prompt

**Learning objective:** Inspect the geographic coverage of the active table.

**Check after execution:** Does the active source cover only Trentino or all Italy?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. List the unique province values in pharmacies. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies.province.unique()

### geodataframe creation from the dataframe with x (longitude) and y (latitude)

GeoDataFrame [constructor](https://geopandas.org/reference/geopandas.GeoDataFrame.html):
* a dataframe (or dictionary)
* the CRS
* a geometry field expressed in WKT (eg. POINT (1,3))



now we are ready to create the geodataframe.
the operation are:
- creation of a geometry column based on the WKT syntax
- transform the DataFrame in GeoDataFrame

### Block 17 · Learning goal and AI prompt

**Learning objective:** Construct point geometry from tabular coordinates with an explicit CRS assumption.

**Check after execution:** Is the coordinate order correct, and what supports the CRS assumption?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Create geo_pharmacies from pharmacies_with_coordinates using longitude as x and latitude as y, assigning EPSG:4326 under the stated exercise assumption. Preserve all attributes and display pharmacy_id, municipality and geometry for the first rows. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
geo_pharmacies = gpd.GeoDataFrame(
    pharmacies_with_coordinates.copy(), crs="EPSG:4326",
    geometry=gpd.points_from_xy(pharmacies_with_coordinates.longitude,
                                pharmacies_with_coordinates.latitude))
geo_pharmacies[["pharmacy_id", "municipality", "geometry"]].head()

### Block 18 · Learning goal and AI prompt

**Learning objective:** Inspect the overall point distribution before integration.

**Check after execution:** Do outliers or coincident points become visible at this scale?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot geo_pharmacies in a 12 by 12 inch figure and show it. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
geo_pharmacies.plot(figsize=(12,12))
plt.show()

### Block 19 · Learning goal and AI prompt

**Learning objective:** Inspect the polygon reference layer separately.

**Check after execution:** What CRS and extent does this standalone plot use?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot macroregions and display the figure. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
macroregions.plot()
plt.show()

overlay the layers

### Block 20 · Learning goal and AI prompt

**Learning objective:** Overlay layers in a common coordinate reference system.

**Check after execution:** Were both layers actually expressed in the same CRS?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot macroregions transformed to EPSG:4326 with black outlines and no fill, then plot geo_pharmacies in green on the same axes with small markers. Show the figure. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
# Align CRS before plotting: GeoPandas does not automatically reproject this overlay.
ax = macroregions.to_crs(4326).plot(edgecolor="black", facecolor="none", figsize=(12, 9))
geo_pharmacies.plot(ax=ax, color="green", markersize=1)
plt.show()

Always align CRS before superimposing these layers. The ISTAT source uses EPSG:32632; pharmacy points in this exercise use EPSG:4326.

### Block 21 · Learning goal and AI prompt

**Learning objective:** Check the source polygon CRS explicitly.

**Check after execution:** Is WGS84 here geographic longitude/latitude or a projected UTM system?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display macroregions.crs. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
macroregions.crs

overlay the layers by using the same projection

### Block 22 · Learning goal and AI prompt

**Learning objective:** Reproduce a correctly aligned point–polygon plot.

**Check after execution:** Does the overlay agree with the separate CRS checks?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot macroregions transformed to EPSG:4326 without fill and overlay geo_pharmacies in green using the same axes, then show the figure. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
ax = macroregions.to_crs(epsg=4326).plot(edgecolor='k', facecolor='none', figsize=(15, 10))
geo_pharmacies.plot(ax=ax,color="green")
plt.show()

---
# Spatial relationships
## how two spatial objects relate to each other

![](https://upload.wikimedia.org/wikipedia/commons/5/55/TopologicSpatialRelarions2.png)

from [https://en.wikipedia.org/wiki/Spatial_relation](https://en.wikipedia.org/wiki/Spatial_relation)

## Relationships between individual objects
Is a selected pharmacy point located in the North-East macroregion?

we need the north-east italian macro-region in wgs84

### Block 23 · Learning goal and AI prompt

**Learning objective:** Identify a stable attribute for selecting a region.

**Check after execution:** Why is a code-based filter safer than assuming row order?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display macroregions before choosing the North-East record. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
macroregions

### Block 24 · Learning goal and AI prompt

**Learning objective:** Retrieve a specific polygon by its administrative code.

**Check after execution:** Is there exactly one matching macroregion?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display the source-CRS macroregion geometry whose COD_RIP is 2, selecting the first matching geometry explicitly. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
macroregions.loc[macroregions.COD_RIP.eq(2), "geometry"].iloc[0]

### Block 25 · Learning goal and AI prompt

**Learning objective:** Prepare a polygon in the same CRS as the pharmacy points.

**Check after execution:** Can northeast now be compared with geo_pharmacies without mixing CRS?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select macroregions where COD_RIP equals 2, transform the selection to EPSG:4326 and store its first geometry as northeast. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
northeast = macroregions.loc[macroregions.COD_RIP.eq(2)].to_crs(4326).geometry.iloc[0]

### Block 26 · Learning goal and AI prompt

**Learning objective:** Inspect a single geometry separately from its table.

**Check after execution:** Which attributes were lost when extracting only the geometry?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display northeast. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
northeast

## let's start with just one point



Select an observed pharmacy record whose declared municipality is Trento. Check its position instead of assuming the address and coordinates agree.

### Block 27 · Learning goal and AI prompt

**Learning objective:** Inspect candidate local pharmacy observations.

**Check after execution:** Which field gives the declared location and which fields give the observed coordinates?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Show the first records of geo_pharmacies whose province_code padded to three digits equals 022. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
geo_pharmacies[geo_pharmacies.province_code.str.zfill(3).eq("022")].head()

we choose the first line and extrac the geometry

### Block 28 · Learning goal and AI prompt

**Learning objective:** Choose one observed point for a relationship test.

**Check after execution:** Does selecting by declared municipality guarantee coordinate correctness?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select pharmacies whose municipality label matches Trento case-insensitively and extract the first geometry as geo_pharmacies_in_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
geo_pharmacies_in_trento = geo_pharmacies.loc[geo_pharmacies.municipality.str.casefold().eq("trento")].geometry.iloc[0]

### Block 29 · Learning goal and AI prompt

**Learning objective:** Inspect the geometry chosen for a predicate test.

**Check after execution:** What information does a Point retain about the pharmacy record?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display geo_pharmacies_in_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
geo_pharmacies_in_trento

## within relation
in our case:<br>&nbsp;&nbsp;&nbsp;&nbsp;it's the point inside the area?

### Block 30 · Learning goal and AI prompt

**Learning objective:** Interpret within as a relationship between two geometries.

**Check after execution:** How would a point on the boundary behave?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Evaluate whether geo_pharmacies_in_trento is within northeast. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
geo_pharmacies_in_trento.within(northeast)

## contain relation
in our case:<br>&nbsp;&nbsp;&nbsp;&nbsp;does the area contain the point?

### Block 31 · Learning goal and AI prompt

**Learning objective:** Understand the inverse relation between contains and within.

**Check after execution:** Should this agree with the preceding within test?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Evaluate whether northeast contains geo_pharmacies_in_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
northeast.contains(geo_pharmacies_in_trento)

we can iterate the operation for each point

_very slow!_

.. so we work with an only one point

### Block 32 · Learning goal and AI prompt

**Learning objective:** Apply a predicate to a collection of observations.

**Check after execution:** Does the selected set follow coordinates or declared province labels?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select all geo_pharmacies points within northeast into a copy named pharmacies_northeast. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies_northeast = geo_pharmacies[geo_pharmacies.within(northeast)].copy()

### Block 33 · Learning goal and AI prompt

**Learning objective:** Read the size of a spatially selected table.

**Check after execution:** Which number counts observations and which counts columns?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display pharmacies_northeast.shape. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies_northeast.shape

### Block 34 · Learning goal and AI prompt

**Learning objective:** Compare declared attributes after a spatial selection.

**Check after execution:** Would an unexpected name prove a location error by itself?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. List unique province values in pharmacies_northeast. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies_northeast.province.unique()

## Different pharmacies, coincident coordinates
The old charging-station “Plaisance” example does not apply to this source. Investigate coordinate repetition among active Trentino pharmacies instead. Repetition is a flag, not a reason to delete records.

### Block 35 · Learning goal and AI prompt

**Learning objective:** Detect suspicious coordinate repetition without deleting observations.

**Check after execution:** Can different pharmacy IDs legitimately or incorrectly share one coordinate pair?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select Trentino pharmacy points into tn_points using province_code 022. Group by exact latitude and longitude, aggregating row count as rows and distinct pharmacy_id count as pharmacy_ids. Store coordinate_groups sorted by descending pharmacy_ids and display its first ten groups. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
tn_points = geo_pharmacies[geo_pharmacies.province_code.str.zfill(3).eq("022")].copy()
coordinate_groups = (tn_points.groupby(["latitude", "longitude"])
    .agg(rows=("record_id", "size"), pharmacy_ids=("pharmacy_id", "nunique"))
    .sort_values("pharmacy_ids", ascending=False))
coordinate_groups.head(10)

### Block 36 · Learning goal and AI prompt

**Learning objective:** Distinguish repeated coordinates from repeated pharmacy identity.

**Check after execution:** Why use distinct IDs rather than only row count?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Keep groups in coordinate_groups with more than one distinct pharmacy ID as coincident_groups and display them. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
coincident_groups = coordinate_groups[coordinate_groups.pharmacy_ids.gt(1)]
coincident_groups

### Block 37 · Learning goal and AI prompt

**Learning objective:** Inspect the records behind an anomaly flag.

**Check after execution:** Do the addresses support a shared location?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. If coincident_groups is nonempty, extract its first latitude/longitude pair and select the matching tn_points into coincident. Otherwise create an empty subset with the same schema. Display pharmacy_id, address, municipality, latitude and longitude. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
if not coincident_groups.empty:
    lat, lon = coincident_groups.index[0]
    coincident = tn_points[tn_points.latitude.eq(lat) & tn_points.longitude.eq(lon)].copy()
else:
    coincident = tn_points.iloc[0:0].copy()
coincident[["pharmacy_id", "address", "municipality", "latitude", "longitude"]]

### Block 38 · Learning goal and AI prompt

**Learning objective:** Locate flagged observations in their administrative context.

**Check after execution:** Can several records be hidden under one visible marker?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot the 2026 Trentino municipality boundaries in EPSG:4326, overlay tn_points in green and coincident points in larger red markers when present. Use the same axes and show the figure. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
ax = municipalities[municipalities.COD_PROV.eq(22)].to_crs(4326).plot(
    edgecolor="gray", facecolor="none", figsize=(10, 10))
tn_points.plot(ax=ax, color="green", markersize=8)
if not coincident.empty:
    coincident.plot(ax=ax, color="red", markersize=80)
plt.show()

### Block 39 · Learning goal and AI prompt

**Learning objective:** Count distinct entities in an anomaly group.

**Check after execution:** Does this equal the number of coordinate pairs?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Calculate the number of distinct pharmacy_id values in coincident. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
coincident.pharmacy_id.nunique()

### Block 40 · Learning goal and AI prompt

**Learning objective:** Maintain a traceable link from anomaly flags to source records.

**Check after execution:** Can every flagged point be traced back to its original CSV row?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Store coincident.record_id as flagged_record_ids and show the matching rows in tn_points. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
flagged_record_ids = coincident.record_id
tn_points[tn_points.record_id.isin(flagged_record_ids)]

### Block 41 · Learning goal and AI prompt

**Learning objective:** Compare the descriptive evidence for coincident observations.

**Check after execution:** What further source evidence would justify changing a coordinate?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display pharmacy_id, address and municipality for coincident. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
coincident[["pharmacy_id", "address", "municipality"]]

### Block 42 · Learning goal and AI prompt

**Learning objective:** Choose an investigation point while handling an empty anomaly set.

**Check after execution:** Does this fallback create or correct any source coordinates?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Set point_to_investigate to the first coincident geometry if available, otherwise to geo_pharmacies_in_trento. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
point_to_investigate = geo_pharmacies_in_trento if coincident.empty else coincident.geometry.iloc[0]

### Block 43 · Learning goal and AI prompt

**Learning objective:** Locate an observed point using polygon containment.

**Check after execution:** Can a wrong location still be inside a valid municipal polygon?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Transform municipalities to EPSG:4326 for a contains test against point_to_investigate, then show PRO_COM_T and COMUNE for the matching source rows. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
municipalities[municipalities.to_crs(4326).contains(point_to_investigate)][["PRO_COM_T", "COMUNE"]]

### Block 44 · Learning goal and AI prompt

**Learning objective:** Cross-check the relevant administrative code and name.

**Check after execution:** Are pharmacy province codes formatted compatibly with ISTAT numeric codes?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display COD_PROV and DEN_PROV for provinces with COD_PROV 22. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
provinces[provinces.COD_PROV.eq(22)][["COD_PROV", "DEN_PROV"]]

A point can be inside a municipality while its actual pharmacy address is elsewhere. Compare IDs, addresses, source dates and coordinate provenance. Do not silently geocode, move or drop the records.

### Block 45 · Learning goal and AI prompt

**Learning objective:** Combine attribute evidence and an interactive map.

**Check after execution:** Can the tooltip reveal multiple records hidden at one position?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. If coincident is nonempty, display its pharmacy IDs, addresses, municipalities and geometries, then create an explore map with those descriptive fields as tooltips. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
if not coincident.empty:
    display(coincident[["pharmacy_id", "address", "municipality", "geometry"]])
    display(coincident.explore(tooltip=["pharmacy_id", "address", "municipality"]))



---


<b>REFERENCE</b>: <br>

Overview of the different functions to check spatial relationships (*spatial predicate functions*):
<ul>
    <li>`equals`</li>
<li>`contains`</li>
<li>`crosses`</li>
<li>`disjoint`</li>
<li>`intersects`</li>
<li>`overlaps`</li>
<li>`touches`</li>
<li>`within`</li>
<li>`covers`</li>
</ul>


See https://shapely.readthedocs.io/en/stable/manual.html#predicates-and-relationships for an overview of those methods.


See https://en.wikipedia.org/wiki/DE-9IM for all details on the semantics of those operations.

## Investigation lab: find pharmacies at exactly the same coordinates

Several records can be hidden behind a single map marker. Here “coincident” means **exactly equal parsed longitude and latitude**, not merely nearby. Do not round coordinates, remove records or move points to make them visible.

Use the active Trentino records already selected at `REFERENCE_DATE`. Keep `record_id`, pharmacy IDs, source coordinates and addresses so every flag can be traced to the original national CSV. Missing-coordinate records remain in the separate audit table; they cannot enter this coordinate comparison.

**Map colours:** green = other mapped pharmacies; red = the selected coincident group; grey/blue = administrative context. On the interactive map, clicking the cluster at maximum zoom spreads the markers for inspection only; the stored coordinates remain unchanged.


### A · Find every repeated location

**Learning objective:** Separate repeated rows, repeated identifiers and different entities sharing one coordinate pair.

**Check after execution:** Does a group contain several distinct IDs, several addresses, or just repeated rows?

**AI prompt — copy into Colab:**

> Group tn_points by exact latitude and longitude. Report row count, distinct pharmacy IDs, distinct addresses and distinct declared municipalities. Keep only groups with more than one distinct pharmacy ID. Preserve all input rows.


In [ ]:
location_audit = (tn_points.groupby(["latitude", "longitude"], dropna=False)
    .agg(source_rows=("record_id", "size"),
         distinct_pharmacies=("pharmacy_id", "nunique"),
         distinct_addresses=("address", "nunique"),
         declared_municipalities=("municipality", "nunique"))
    .reset_index()
    .sort_values(["distinct_pharmacies", "latitude", "longitude"],
                 ascending=[False, True, True]))
suspicious_locations = location_audit.query("distinct_pharmacies > 1").copy()
display(suspicious_locations)
print("Coincident groups:", len(suspicious_locations))
print("Records without usable coordinates:", len(tn_records) - len(tn_points))


### B · Inspect the records behind a flag

**Learning objective:** Compare identity and address evidence without calling coincident records duplicates.

**Check after execution:** For the 22 October 2026 snapshot the largest group contains seven distinct pharmacy IDs.

**AI prompt — copy into Colab:**

> Select the first suspicious location if any exists, retaining exact coordinates. Display source record ID, pharmacy ID, address, municipality, validity dates and coordinates. Handle an empty result explicitly.


In [ ]:
if suspicious_locations.empty:
    selected_group = tn_points.iloc[0:0].copy()
    print("No coincident group found for this source snapshot.")
else:
    selected_location = suspicious_locations.iloc[0]
    selected_group = tn_points.loc[
        tn_points.latitude.eq(selected_location.latitude)
        & tn_points.longitude.eq(selected_location.longitude)].copy()
    display(selected_group[["record_id", "pharmacy_id", "address", "municipality",
                            "data_inizio_validita", "data_fine_validita",
                            "latitude", "longitude"]].sort_values("pharmacy_id"))
    print("Rows:", len(selected_group))
    print("Distinct pharmacy IDs:", selected_group.pharmacy_id.nunique())


### C · Make overlapping records visible as an anomaly

**Learning objective:** Show the observed location in local projected context; explain why seven records produce one visible point.

**Check after execution:** The red ring is one coordinate pair. Its label counts distinct IDs, not visible markers. This map alone cannot establish whether a point is in a river.

**AI prompt — copy into Colab:**

> Plot a 1.5 km wide window around the selected coincident group in EPSG:32632. Show municipality boundaries, other pharmacy points in green and the selected location as a red ring labelled with its distinct ID count. Do not jitter or alter source coordinates.


In [ ]:
if not selected_group.empty:
    from shapely.geometry import box
    selected_projected = selected_group.to_crs(32632)
    selected_point = selected_projected.geometry.iloc[0]
    x, y = selected_point.x, selected_point.y
    view = box(x - 750, y - 750, x + 750, y + 750)
    local_boundaries = municipalities[municipalities.COD_PROV.eq(22)].to_crs(32632).clip(view)
    other_points = tn_points.loc[~tn_points.record_id.isin(selected_group.record_id)].to_crs(32632)
    other_points = other_points.loc[other_points.intersects(view)]
    fig, ax = plt.subplots(figsize=(8, 7))
    local_boundaries.plot(ax=ax, facecolor="#edf1f4", edgecolor="#004c85", linewidth=0.8)
    if not other_points.empty:
        other_points.plot(ax=ax, color="green", markersize=45, label="Other pharmacies")
    ax.scatter([x], [y], facecolors="none", edgecolors="red", s=250,
               linewidths=2.5, label="Selected coincident group", zorder=5)
    ax.annotate(f"{selected_group.pharmacy_id.nunique()} distinct IDs; one location",
                (x, y), xytext=(12, 15), textcoords="offset points", color="red")
    ax.set(xlim=(x-750,x+750), ylim=(y-750,y+750),
           xlabel="Easting (m) — EPSG:32632", ylabel="Northing (m) — EPSG:32632",
           title="Investigate a shared coordinate; do not infer a correction")
    ax.set_aspect("equal"); ax.ticklabel_format(style="plain", useOffset=False)
    ax.legend(loc="lower left"); plt.tight_layout(); plt.show()


### D · Open every hidden record on an interactive map

**Learning objective:** Inspect overlapping records individually while preserving their actual positions.

**Check after execution:** Zoom to the maximum and click the numbered cluster to spread its markers; this is a display operation, not a positional correction. The basemap requires internet access.

**AI prompt — copy into Colab:**

> Create a Folium map of the selected coincident pharmacies. Use MarkerCluster with spiderfyOnMaxZoom enabled, red markers and escaped popup text containing source record ID, pharmacy ID, address and declared municipality. Show other nearby pharmacies in green. Save the HTML without modifying any coordinates.


In [ ]:
if not selected_group.empty:
    import folium
    from folium.plugins import MarkerCluster
    from html import escape
    map_lat, map_lon = selected_group[["latitude", "longitude"]].iloc[0]
    coincidence_map = folium.Map(location=[map_lat, map_lon], zoom_start=18,
                                max_zoom=19, tiles="OpenStreetMap")
    hidden_records = MarkerCluster(name="Coincident records: zoom in and click",
        options={"spiderfyOnMaxZoom": True, "showCoverageOnHover": False}).add_to(coincidence_map)
    for _, row in selected_group.iterrows():
        popup = "<br>".join(f"{label}: {escape(str(row[field]))}" for label, field in [
            ("Source row", "record_id"), ("Pharmacy ID", "pharmacy_id"),
            ("Address", "address"), ("Declared municipality", "municipality")])
        folium.Marker([row.latitude, row.longitude], popup=folium.Popup(popup, max_width=350),
            tooltip=f"Pharmacy {row.pharmacy_id}", icon=folium.Icon(color="red")).add_to(hidden_records)
    nearby_layer = folium.FeatureGroup(name="Other nearby pharmacies").add_to(coincidence_map)
    for _, row in other_points.to_crs(4326).iterrows():
        folium.CircleMarker([row.geometry.y, row.geometry.x], radius=5,
            color="green", fill=True, fill_opacity=0.8,
            tooltip=escape(str(row.pharmacy_id))).add_to(nearby_layer)
    folium.LayerControl().add_to(coincidence_map)
    coincidence_html = DATA / "pharmacy_coincident_records.html"
    coincidence_map.save(str(coincidence_html))
    display(coincidence_map)
    print("Saved interactive map:", coincidence_html)


### E · Export flags, not corrections

**Learning objective:** Make a review table traceable to the original data and snapshot.

**Check after execution:** No coordinate, ID or address in tn_points has been changed. A flag does not establish an error.

**AI prompt — copy into Colab:**

> Merge every suspicious coordinate pair back to tn_points. Export a CSV with source identity, addresses, original coordinates, reference date and a review flag. Do not delete, relocate or geocode any record.


In [ ]:
flagged_pharmacies = tn_points.merge(
    suspicious_locations[["latitude", "longitude", "distinct_pharmacies"]],
    on=["latitude", "longitude"], how="inner", validate="many_to_one")
flagged_pharmacies["review_flag"] = "different_ids_same_coordinates"
flagged_pharmacies["reference_date"] = REFERENCE_DATE.strftime("%Y-%m-%d")
review_columns = ["record_id", "pharmacy_id", "address", "municipality", "latitude",
                  "longitude", "distinct_pharmacies", "reference_date", "review_flag"]
flagged_pharmacies[review_columns].to_csv(DATA / "pharmacy_coordinate_review.csv", index=False)
display(flagged_pharmacies[review_columns])


### Discuss before correcting

What evidence supports a genuinely shared location? Could the coordinates be an address fallback or a geocoding artefact? Are validity dates, pharmacy IDs and administrative codes aligned across sources? What authoritative address or coordinate source would justify a correction?

A point inside the correct municipality can still be misplaced. The basemap is contextual evidence and is not, by itself, proof of a pharmacy position. Keep observed coordinates and any later verified correction in separate columns with provenance.


# Spatial Joins

You can create a join like the usual [join](https://pandas.pydata.org/pandas-docs/stable/user_guide/merging.html) between pandas dataframe by using a spatial relationship with the function [geopandas.sjoin](http://geopandas.readthedocs.io/en/latest/reference/geopandas.sjoin.html)

### Block 46 · Learning goal and AI prompt

**Learning objective:** Transfer attributes through an explicit spatial relationship.

**Check after execution:** Which geometry survives, and are unmatched pharmacy records retained?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Perform a left GeoPandas spatial join with geo_pharmacies on the left and macroregions transformed to EPSG:4326 on the right, keeping COD_RIP, DEN_RIP and geometry from the polygons. Use predicate within and store pharmacies_and_macroregions. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies_and_macroregions = gpd.sjoin(
    geo_pharmacies, macroregions.to_crs(4326)[["COD_RIP", "DEN_RIP", "geometry"]],
    how="left", predicate="within")

### Block 47 · Learning goal and AI prompt

**Learning objective:** Inspect how a join changes a schema.

**Check after execution:** Which attributes came from the point layer and which from polygons?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display pharmacies_and_macroregions.columns. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies_and_macroregions.columns

### Block 48 · Learning goal and AI prompt

**Learning objective:** Audit spatial-join cardinality per source record.

**Check after execution:** Would counting output rows reveal lost or multiply matched source records?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Group pharmacies_and_macroregions by record_id and count non-null COD_RIP matches as matches_per_record. Show the distribution and separately print counts with zero and more than one match. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
matches_per_record = pharmacies_and_macroregions.groupby("record_id").COD_RIP.count()
print(matches_per_record.value_counts().sort_index())
print("Unmatched records:", matches_per_record.eq(0).sum())
print("Multiple matches:", matches_per_record.gt(1).sum())

### Block 49 · Learning goal and AI prompt

**Learning objective:** Read sample joined records without treating them as validation of all rows.

**Check after execution:** Can a correct-looking sample conceal unmatched or ambiguous records?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display the first five rows of pharmacies_and_macroregions. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies_and_macroregions.head(5)

### Block 50 · Learning goal and AI prompt

**Learning objective:** Verify the geometry retained by a left spatial join.

**Check after execution:** Why are the retained geometries points rather than macroregion polygons?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display unique geometry types in pharmacies_and_macroregions. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies_and_macroregions.geom_type.unique()

... and now you can investigate the new geodataframe

### Block 51 · Learning goal and AI prompt

**Learning objective:** Count distinct entities after integration.

**Check after execution:** How are unmatched points and IDs matched to several regions handled?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Group pharmacies_and_macroregions by DEN_RIP and count distinct pharmacy_id values. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies_and_macroregions.groupby("DEN_RIP").pharmacy_id.nunique()

### Block 52 · Learning goal and AI prompt

**Learning objective:** Present a grouped result with an explicit counting unit.

**Check after execution:** Does the chart make missing or ambiguous matches visible?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot distinct pharmacy_id counts by DEN_RIP from pharmacies_and_macroregions as a bar chart, label the vertical axis Distinct pharmacy IDs and show the figure. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacies_and_macroregions.groupby("DEN_RIP").pharmacy_id.nunique().plot(kind="bar")
plt.ylabel("Distinct pharmacy IDs")
plt.show()


<b>SPATIAL JOIN </b>= *transferring attributes from one layer to another based on their spatial relationship*<br/>

Different parts of this operations:
<ul>
    <li>The GeoDataFrame to which we want add information</li>
    <li>The GeoDataFrame that contains the information we want to add</li>
    <li>The spatial relationship we want to use to match both datasets ('intersects', 'contains', 'within')</li>
    <li>The type of join: left or inner join</li>
</ul>


## Municipality join and audit
Keep the declared code and compare it with the spatially assigned code. A left join preserves unmatched points; it does not resolve multiple matches.

### Block 53 · Learning goal and AI prompt

**Learning objective:** Compare a declared administrative assignment with a spatial assignment.

**Check after execution:** Could different dates, code conventions or inaccurate coordinates explain a disagreement?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select COD_PROV 22 municipalities as tn_municipalities. Left-join tn_points to their polygons after CRS alignment using within, preserving PRO_COM_T and COMUNE, as joined_tn. Count matches per record_id as match_count. Pad declared municipality_code and spatial PRO_COM_T to six-character strings, preserving missing values, as declared_code and spatial_code. Select nonmissing disagreements as code_disagreements and display IDs, addresses and both codes. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
tn_municipalities = municipalities[municipalities.COD_PROV.eq(22)].copy()
joined_tn = gpd.sjoin(tn_points, tn_municipalities.to_crs(tn_points.crs)[
    ["PRO_COM_T", "COMUNE", "geometry"]], how="left", predicate="within")
match_count = joined_tn.groupby("record_id").PRO_COM_T.count()
print("Matches per source record:\n", match_count.value_counts().sort_index())
joined_tn["declared_code"] = joined_tn.municipality_code.str.zfill(6)
joined_tn["spatial_code"] = joined_tn.PRO_COM_T.astype("string").str.zfill(6)
code_disagreements = joined_tn[joined_tn.spatial_code.notna()
    & joined_tn.declared_code.ne(joined_tn.spatial_code)]
code_disagreements[["pharmacy_id", "address", "declared_code", "spatial_code"]]

### Block 54 · Learning goal and AI prompt

**Learning objective:** Produce counts while making exclusions and unresolved matches visible.

**Check after execution:** Does zero mean no real pharmacy or no counted pharmacy under these rules?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Create resolved_tn using only records with exactly one municipal match, without arbitrarily resolving ambiguous points. Count distinct pharmacy_id by spatial_code, merge counts back to all tn_municipalities as summary_tn and fill absent counts with zero. Plot pharmacy_count and report active Trentino records without usable coordinates, unmatched mapped records and ambiguous mapped records. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
# Do not silently choose one municipality for an ambiguous point.
resolved_tn = joined_tn[joined_tn.record_id.isin(match_count[match_count.eq(1)].index)]
counts = resolved_tn.groupby("spatial_code").pharmacy_id.nunique().rename("pharmacy_count")
summary_tn = tn_municipalities.merge(counts, left_on="PRO_COM_T", right_index=True, how="left")
summary_tn["pharmacy_count"] = summary_tn.pharmacy_count.fillna(0).astype(int)
summary_tn.plot(column="pharmacy_count", legend=True, edgecolor="gray", figsize=(10, 9))
plt.show()
print("Unmapped active source records:", len(tn_records) - len(tn_points))
print("Unmatched mapped records:", match_count.eq(0).sum())
print("Ambiguous mapped records:", match_count.gt(1).sum())

---
# Spatial operations
GeoPandas provide analysis methods that return new geometric objects (based on shapely)

See [https://shapely.readthedocs.io/en/stable/manual.html#spatial-analysis-methods](https://shapely.readthedocs.io/en/stable/manual.html#spatial-analysis-methods) for more details.

## buffer
*object.buffer(distance, resolution=16, cap_style=1, join_style=1, mitre_limit=5.0)*

Returns an approximate representation of all points within a given distance of the this geometric object.

![](https://raw.githubusercontent.com/napo/geospatial_course_unitn/master/images/buffer.png)


### Block 55 · Learning goal and AI prompt

**Learning objective:** Use metre-based coordinates before constructing a distance buffer.

**Check after execution:** Is the point’s coordinate quality established before measuring from it?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select the first pharmacy whose declared municipality is Trento, transform its GeoDataFrame to EPSG:32632 and extract its geometry as pharmacy_in_trento_32632. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacy_in_trento_32632 = geo_pharmacies.loc[geo_pharmacies.municipality.str.casefold().eq("trento")].to_crs(32632).geometry.iloc[0]

### Block 56 · Learning goal and AI prompt

**Learning objective:** Inspect the projected point used for a distance operation.

**Check after execution:** Are these coordinates metres rather than degrees?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display pharmacy_in_trento_32632. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacy_in_trento_32632

### Block 57 · Learning goal and AI prompt

**Learning objective:** Interpret a buffer as a geometric proximity area.

**Check after execution:** Why is this not a walking or driving accessibility area?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Create a buffer of 9000 metres around pharmacy_in_trento_32632. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
pharmacy_in_trento_32632.buffer(9000) # a circle with a ray of 9000 meters

due to the algorithm with which the buffer is built, as the value increases, from whatever geometry one starts, the result will take on more and more the shape of a circumference.


<img src="https://raw.githubusercontent.com/napo/geospatial_course_unitn/master/images/buffer.gif" width="450px"/>

## simplify

*object.simplify(tolerance, preserve_topology=True)*

    Returns a simplified representation of the geometric object.


### Block 58 · Learning goal and AI prompt

**Learning objective:** Prepare a polygon in the CRS used by subsequent operations.

**Check after execution:** Do polygon and pharmacy buffer now use compatible units?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select macroregions with COD_RIP 2, transform to EPSG:32632 and store its first geometry as northeast_geometry. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
northeast_geometry = macroregions[macroregions.COD_RIP.eq(2)].to_crs(32632).geometry.iloc[0]

### Block 59 · Learning goal and AI prompt

**Learning objective:** Inspect the polygon before simplifying it.

**Check after execution:** Which small boundary details may be lost in a simplified representation?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display northeast_geometry. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
northeast_geometry

### Block 60 · Learning goal and AI prompt

**Learning objective:** Recognise that simplification changes the represented boundary.

**Check after execution:** What does this setting preserve, and what detail can still change?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Simplify northeast_geometry with a 10000 metre tolerance and preserve_topology=True. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
northeast_geometry.simplify(10000, preserve_topology=True)

## Es. symmetric_difference
*object.symmetric_difference(other)*

    Returns a representation of the points in this object not in the other geometric object, and the points in the other not in this geometric object.


### Block 61 · Learning goal and AI prompt

**Learning objective:** Interpret symmetric difference as membership in exactly one geometry.

**Check after execution:** Which areas belong to the result and which are removed?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Take northeast_geometry simplified with a 10000 metre tolerance and preserved topology, and compute its symmetric difference with the 9000 metre buffer around pharmacy_in_trento_32632. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
northeast_geometry.simplify(10000, preserve_topology=True).symmetric_difference(pharmacy_in_trento_32632.buffer(9000))

<b>REMEMBER:</b>

GeoPandas (and Shapely for the individual objects) provides a whole lot of basic methods to analyse the geospatial data (distance, length, centroid, boundary, convex_hull, simplify, transform, ....), much more than the few that we can touch in this tutorial.
<ul>
<li>An overview of all methods provided by GeoPandas can be found here: <a href="https://geopandas.readthedocs.io/en/latest/docs/reference.html">https://geopandas.readthedocs.io/en/latest/docs/reference.html</a></li>
    </ul>



---
# Clip

Extracts input features that overlay the clip features.

![](https://desktop.arcgis.com/en/arcmap/10.3/tools/analysis-toolbox/GUID-6D3322A8-57EA-4D24-9FFE-2A9E7C6B29EC-web.gif)

```python
 geopandas.clip(gdf, mask, keep_geom_type=False)
 ```
 *gdf* = geodataframe<Br/>
 *mask* = polygon

### Block 62 · Learning goal and AI prompt

**Learning objective:** Clip features rather than merely selecting whole features.

**Check after execution:** Are output geometries whole municipalities or truncated pieces?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Transform municipalities to EPSG:32632 and clip them with the 9000 metre buffer around pharmacy_in_trento_32632, storing municipalities_inside_circle. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
municipalities_inside_circle = municipalities.to_crs(32632).clip(pharmacy_in_trento_32632.buffer(9000))

### Block 63 · Learning goal and AI prompt

**Learning objective:** Identify observations represented by a clipped result.

**Check after execution:** Does appearing in this list mean the entire municipality lies inside the circle?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. List unique COMUNE values in municipalities_inside_circle. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
municipalities_inside_circle.COMUNE.unique()

### Block 64 · Learning goal and AI prompt

**Learning objective:** Read newly created clipped geometries on a map.

**Check after execution:** Where does the circular mask truncate the boundaries?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot municipalities_inside_circle in a 10 by 10 inch figure and show it. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
municipalities_inside_circle.plot(figsize=(10,10))
plt.show()

---
# Aggregation with dissolve

Spatial data are often more granular than we need. For example, we have the data of the macro-regions but we don't have a geometry with the border of Italy.

If we have a columns to operate a *groupby* we can solve it but to create the geometry we need the function *dissolve*.



### Block 65 · Learning goal and AI prompt

**Learning objective:** Create a grouping key for spatial aggregation.

**Check after execution:** Does assigning a common label already merge the geometries?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Add a nation column with the value italy to every macroregions row. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
macroregions['nation']='italy'

### Block 66 · Learning goal and AI prompt

**Learning objective:** Inspect the table after adding a grouping attribute.

**Check after execution:** Are the original separate geometries still present?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display macroregions. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
macroregions

### Block 67 · Learning goal and AI prompt

**Learning objective:** Choose attributes intentionally before dissolving.

**Check after execution:** Which other attributes are excluded from aggregation?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Create italy containing only the nation and geometry columns of macroregions. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
italy = macroregions[['nation', 'geometry']]

### Block 68 · Learning goal and AI prompt

**Learning objective:** Establish the visual state before dissolving.

**Check after execution:** Are internal macroregion borders still present?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot italy and explicitly show the figure. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
italy.plot()
plt.show()

### Block 69 · Learning goal and AI prompt

**Learning objective:** Combine geometry union with grouped tabular aggregation.

**Check after execution:** How many grouped features remain, and what happened to the index?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Transform italy to EPSG:4326 and dissolve it by nation, replacing italy with the result. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:

italy = italy.to_crs(epsg=4326).dissolve(by='nation')
%time

### Block 70 · Learning goal and AI prompt

**Learning objective:** Inspect the output of a spatial groupby.

**Check after execution:** How did row count and index differ from the original table?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display the dissolved italy table. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
italy

### Block 71 · Learning goal and AI prompt

**Learning objective:** Retrieve geometry by position after the index has changed.

**Check after execution:** Why might geometry[0] fail when the index is a nation label?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Display the first geometry of italy using explicit positional indexing. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
italy.geometry.iloc[0]

### Block 72 · Learning goal and AI prompt

**Learning objective:** Transform an aggregated geometry for projected analysis.

**Check after execution:** Does reprojection split the dissolved feature back into macroregions?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Transform italy to EPSG:32632 and display its first geometry by position. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
italy.to_crs(epsg=32632).geometry.iloc[0]

<b>REMEMBER:</b>

dissolve can be thought of as doing three things: (a) it **dissolves** all the geometries within a given group together into a single geometric feature (using the *unary_union* method), and (b) it **aggregates** all the rows of data in a group using *groupby.aggregate()*, and (c) it **combines** those two results.
    
<ul>
<li>An overview of all methods provided by GeoPandas can be found here: <a href="http://geopandas.org/aggregation_with_dissolve.html">http://geopandas.org/aggregation_with_dissolve.html</a></li>
 </ul>

---
# Overlay

Spatial overlays allow you to compare two GeoDataFrames containing polygon or multipolygon geometries and create a new GeoDataFrame with the new geometries representing the spatial combination and merged properties. This allows you to answer questions like

    What are the demographics of the census tracts within 90km from a point?

The basic idea is demonstrated by the graphic below but keep in mind that overlays operate at the dataframe level, not on individual geometries, and the properties from both are retained

![](https://docs.qgis.org/testing/en/_images/overlay_operations.png)

source: https://geopandas.org/gallery/overlays.html


### Block 73 · Learning goal and AI prompt

**Learning objective:** Prepare the right-hand polygon layer for an overlay.

**Check after execution:** Are both overlay inputs in the same CRS?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Select macroregions where COD_RIP equals 2 and transform them to EPSG:32632 as macroregion_gdf. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
macroregion_gdf = macroregions[macroregions.COD_RIP==2].to_crs(epsg=32632)

### Block 74 · Learning goal and AI prompt

**Learning objective:** Interpret directional polygon difference at layer level.

**Check after execution:** Does the result mean Italy minus North-East or the reverse?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Transform italy to EPSG:32632 and perform an overlay difference with macroregion_gdf, storing overlay. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
overlay = italy.to_crs(epsg=32632).overlay(macroregion_gdf, how="difference")

### Block 75 · Learning goal and AI prompt

**Learning objective:** Check an overlay result visually.

**Check after execution:** Does the remaining area agree with the chosen difference direction?

**AI prompt — copy into Colab:**

> Write the next Python cell for this notebook. Plot overlay and show the figure. Reuse existing variables and generate only this step. Do not invent results.


In [ ]:
overlay.plot()
plt.show()

# Exercise: pharmacies and municipalities of Trentino
1. Select the ISTAT 2026 municipalities with COD_PROV 22 and valid pharmacy records at the stated reference date.
2. Perform a **left** point–polygon join and audit zero, one and multiple matches for each record_id.
3. Compare the declared municipality_code with PRO_COM_T, preserving six-character codes.
4. Count distinct pharmacy_id values by municipality. Resolve multiple matches explicitly. Include municipalities with zero matches and report excluded records separately.
5. Find the largest municipality from the 2026 geometries, select its touching neighbours and dissolve them into a macroarea. Check the effect of corner-only contact.
6. For each pharmacy in the macroarea, count distinct other pharmacies within 500 metres in a suitable projected CRS. Exclude the same pharmacy ID from its own count.
7. Compare counts with and without flagged coincident coordinates. Do not describe the excluded scenario as corrected truth.
8. Export the results and an anomaly table with source URLs, dates, predicates and exclusions.

Optional: rebuild Elba using selection, dissolve and overlay, as in the original notebook.
